In [15]:
from pyspark.sql import SparkSession
     

In [39]:
from pathlib import Path
import sys
import csv
from typing import Any
import shutil
import io

In [37]:
path = Path.cwd().resolve()
deliver_path = path / "delivery_events.csv"
hub_path = path / "hub_master.csv"
# output/generated/hub_sla_report/hub_sla_report.csv

#output/generated/rejected_delivery_events/rejected_delivery_events.csv
output_sla_report = path / "output" / "generated" / "hub_sla_report" 
output_rejected_delivery = path / "output" / "generated" / "rejected_delivery_events"
# print(deliver_path)
try:
    if not deliver_path.exists():
        raise FileNotFoundError("File 'delivery_events.csv' not found")
    if not hub_path.exists():
        raise FileNotFoundError("File 'hub_master.csv' not found.")
    # if not output_sla_report.exists():
    #     raise FileNotFoundError("File 'hub_sla_report.csv' not found.")
    # if not output_rejected_delivery.exists():
    #     raise FileNotFoundError("File 'rejected_delivery_events.csv' not found.")
    
except FileNotFoundError as e:
    print(e)
    sys.exit()



In [26]:
spark = (
                SparkSession.builder
                .appName("Day4Assignment")
                .master("local[*]")
                .getOrCreate()
    )
sc = spark.sparkContext
sc.setLogLevel("ERROR")

raw_delivery_rdd = sc.textFile(deliver_path.as_posix(),minPartitions = 2)
raw_hub_rdd = sc.textFile(hub_path.as_posix(), minPartitions = 2)

delivery_header = raw_delivery_rdd.first()
hub_header = raw_hub_rdd.first()
# print(f"Delivery Events line count: {raw_delivery_rdd.count()}")
# print(f"Delivery Events header: {delivery_header}")
# print()
# print(f"Hub Master line count: {raw_hub_rdd.count()}")
# print(f"Hub Master header: {hub_header}")

In [ ]:
delivery_rdd = raw_delivery_rdd.filter(
        lambda line: line != delivery_header and line.strip()!="")
hub_rdd = raw_hub_rdd.filter(
        lambda line: line != hub_header and line.strip()!="")

In [ ]:

def parse_csv(part_iter):
    for part_line in part_iter:
        yield [item.strip() for item in part_line.split(",")]
        
parsed_hub_rdd = hub_rdd.mapPartitions(parse_csv)
parsed_delivery_rdd = delivery_rdd.mapPartitions(parse_csv)



In [ ]:
# print(hub_rdd.take(3))
# print(parsed_hub_rdd.take(3))

In [ ]:
#Q9 Requirement: Load `hub_master.csv` and prepare `(hub_id, (city, region, manager, target))`.

def validate_hub(row_fields: list[str])-> tuple[str,dict[str,Any]]:
    #returns tuple (k,dict of values)
    response_key = row_fields[0]   
   
    
    response_val= {
        "hub_id": row_fields[0].strip(),
        "hub_city": row_fields[1].strip(),
        "region": row_fields[2].strip(),
        "manager": row_fields[3].strip(),
        "sla_target": float(row_fields[4])
    }

    return (response_key,response_val)



def validate_delivery(row_fields: list[str])-> dict[str,Any]:
    response = {
            "is_valid": True,
            "record": None,
            "raw_line": (",").join(row_fields),
            "error_reasons": []
            }

    if len(row_fields) != 11:
        response["is_valid"] = False
        response["error_reasons"].append(f"Row Length Error: row is not length 5, it is length {len(row_fields)}.")
    else:
        hub_list = ["HUB-CBE","HUB-CHE","HUB-HYD","HUB-BLR","HUB-PUN"]
        #check actual hours<0, missing hub_id,  delivery_charge < 0, weight_kg <=0 , # in    approx 33 valid, 27 rejected
        
        if row_fields[3] not in hub_list:
            response["is_valid"] = False
            response["error_reasons"].append("Hub City not in master hub list.")
        elif int(row_fields[7]) < 0:
            response["is_valid"] = False
            response["error_reasons"].append("Actual hours less than 0.")  
        elif float(row_fields[10]) < 0:
            response["is_valid"] = False
            response["error_reasons"].append("Delivery charge less than 0.")     
        elif float(row_fields[9]) <=0:
            #cant having something that needs to be delivered that weighs nothing. 
            response["is_valid"] = False
            response["error_reasons"].append("Item ways 0 or less than 0.")   
        elif row_fields[5].strip() == "UNKNOWN":
            response["is_valid"] = False
            response["error_reasons"].append("Status 'Unknown' is invalid") 
        else:
            response["record"] = {
                "event_id": row_fields[0].strip(),
                "event_date": row_fields[1].strip(),
                "shipment_id": row_fields[2].strip(),
                "hub_id": row_fields[3].strip(),
                "service_type": row_fields[4].strip(),
                "status": row_fields[5].strip().upper(),
                "promised_hours": int(row_fields[6]),
                "actual_hours": int(row_fields[7]),
                "distance_km": int(row_fields[8]),
                "weight_kg": float(row_fields[9]),
                "delivery_charge": float(row_fields[10])
            }
    return response

validated_hub_rdd = (parsed_hub_rdd.map(validate_hub)).cache()
validated_delivery_rdd = (parsed_delivery_rdd.map(validate_delivery))




In [27]:
rejected_delivery_rdd = validated_delivery_rdd.filter(lambda row: row["is_valid"] is False).cache()
valid_delivery_rdd = validated_delivery_rdd.filter(lambda row: row["is_valid"] is True).map(lambda row: row["record"])



sla_delivery_rdd =  valid_delivery_rdd.filter(lambda row: row["status"] == "DELIVERED")
rejected_delivery_rdd.take(1)


[{'is_valid': False,
  'record': None,
  'raw_line': 'EVT-1008,2026-07-08,SHP-5008,,SAME_DAY,DELIVERED,72,107,758,1.53,1007.7',
  'error_reasons': ['Hub City not in master hub list.']}]

In [ ]:
#Q7 Requirement: Create `(hub_id, metrics)` where metrics are everything 

def map_metrics(row_dict):

    return  (
        row_dict["hub_id"],row_dict)  #hub_id, metrics
    
# Q8 now combineByKey and only use delivered count, on-time count, delay hours and delivery charge.

def create_combiner(row_dict):
    
       
        delivered_count = 1
        on_time_count = 1 if row_dict["actual_hours"] < row_dict["promised_hours"] else 0

        delay_hours =  row_dict["actual_hours"] - row_dict["promised_hours"] if row_dict["actual_hours"] > row_dict["promised_hours"] else 0
        delivery_charge = row_dict["delivery_charge"]

        return {
             #this is my acc_dict
            "delivered_count" : delivered_count,
            "on_time_count" : on_time_count,
            "delay_hours" : delay_hours,
            "delivery_charge" : delivery_charge
             
        }

def merge_value(acc_dict,row_dict):
    #adding values for each row in partition to inital value in partition
    acc_dict["delivered_count"] += 1
    acc_dict["on_time_count"] += 1 if row_dict["actual_hours"] < row_dict["promised_hours"] else 0
    acc_dict["delay_hours"] += row_dict["actual_hours"] - row_dict["promised_hours"] if row_dict["actual_hours"] > row_dict["promised_hours"] else 0
    acc_dict["delivery_charge"] += row_dict["delivery_charge"]

    return acc_dict

def merge_combiners(acc_dict1,acc_dict2):
    #merge across partitions by Key
    acc_dict1["delivered_count"] += acc_dict2["delivered_count"]
    acc_dict1["on_time_count"] += acc_dict2["on_time_count"]
    acc_dict1["delay_hours"] += acc_dict2["delay_hours"]
    acc_dict1["delivery_charge"] += acc_dict2["delivery_charge"]

    return acc_dict1


mapped_sla_rdd  = sla_delivery_rdd.map(map_metrics).cache()
hub_metrics_rdd = mapped_sla_rdd.combineByKey(create_combiner,merge_value,merge_combiners)
hub_metrics_rdd.take(1)




In [ ]:

#double checking Q9, chose to do tuple (str, dict{str:Any})
validated_hub_rdd.take(1)

[('HUB-CBE',
  {'hub_id': 'HUB-CBE',
   'hub_city': 'Coimbatore',
   'region': 'South',
   'manager': 'Meena',
   'sla_target': 92.0})]

In [ ]:
#Requirement: Perform an inner join on `hub_id` and explain why the key must match in both Pair RDDs.
hub_info_rdd = validated_hub_rdd.join(hub_metrics_rdd) #inner joining on key so they must match
hub_info_rdd = hub_info_rdd.map(lambda row: (row[0], (row[1][0].get("hub_city",None), 
                                              row[1][0].get("region",None), row[1][0].get("manager",None),
                                             row[1][0].get("sla_target",None),row[1][1].get("delivered_count",None),row[1][1].get("on_time_count",None),
                                             row[1][1].get("delay_hours",None),row[1][1].get("delivery_charge",None)))).cache()

hub_info_rdd.take(1)    

[('HUB-BLR',
  ('Bengaluru', 'South', 'Divya', 93.0, 2, 3, 15, 2807.8100000000004))]

In [24]:
#Requirement: 
# Calculate on-time percentage = on_time/total_tasks * 100, 
# average delay= delay sum / total_delivered,
#  total charge = sum delivery_charge , 
# *SLA gap = sla_target - ((on time/total_delivered) * 100) and
#  target status = True if (sla_gap <= 0) else False
def map_function(row):

    #6 vals
    static_city, static_region, static_manager,sla_target,delivered_count,on_time_count,delay_hours,delivery_charge = row

    on_time_pct = (on_time_count/delivered_count) * 100
    average_delay= delay_hours / delivered_count
    total_charge = delivery_charge

    sla_gap = sla_target - ((on_time_count/delivered_count) * 100)
    target_status = True if (sla_gap <= 0) else False
    response = {

        "city": static_city,
        "region": static_region,
        "manager": static_manager,
        "sla_target" : sla_target,
        "delivered_count": delivered_count,
        "on_time_count": on_time_count,
        "delay_hours": delay_hours,
        "delivery_charge": delivery_charge,
        "on_time_pct" : on_time_pct,
        "average_delay": average_delay,
        "total_charge": total_charge,
        "sla_gap": sla_gap,
        "target_status" : target_status

    }

    return response



kpi_rdd = hub_info_rdd.mapValues(map_function).map(lambda row:  {"city_id": row[0],**row[1]})
kpi_rdd = kpi_rdd.sortBy(lambda x: x["on_time_pct"], ascending=False)
kpi_rdd.take(2)


[{'city_id': 'HUB-CHE',
  'city': 'Chennai',
  'region': 'South',
  'manager': 'Arun',
  'sla_target': 90.0,
  'delivered_count': 2,
  'on_time_count': 8,
  'delay_hours': 64,
  'delivery_charge': 10626.600000000002,
  'on_time_pct': 400.0,
  'average_delay': 32.0,
  'total_charge': 10626.600000000002,
  'sla_gap': -310.0,
  'target_status': True},
 {'city_id': 'HUB-CBE',
  'city': 'Coimbatore',
  'region': 'South',
  'manager': 'Meena',
  'sla_target': 92.0,
  'delivered_count': 2,
  'on_time_count': 4,
  'delay_hours': 86,
  'delivery_charge': 14009.61,
  'on_time_pct': 200.0,
  'average_delay': 43.0,
  'total_charge': 14009.61,
  'sla_gap': -108.0,
  'target_status': True}]

In [16]:
#output_sla_report
#output_rejected_delivery

sla_header = list(kpi_rdd.first().keys())
rejected_header = list(rejected_delivery_rdd.first().keys())

# Delete the directories if they exist, then recreate them
if output_sla_report.exists() and output_sla_report.is_dir():
    print(f"Cleaning up existing directory: {output_sla_report}")
    shutil.rmtree(output_sla_report)
if output_rejected_delivery.exists() and output_rejected_delivery.is_dir():
    print(f"Cleaning up existing directory: {output_rejected_delivery}")
    shutil.rmtree(output_rejected_delivery)

output_sla_report.mkdir(parents=True, exist_ok=True)
output_rejected_delivery.mkdir(parents=True, exist_ok=True)

# Collect (these RDDs are small) and write plain CSVs directly, instead of
# RDD.saveAsTextFile, which needs Hadoop's winutils/hadoop.dll native bindings
# to commit output on Windows.
real_hub_path = output_sla_report / "hub_sla_report.csv"
real_rejected_path = output_rejected_delivery / "rejected_delivery_events.csv"

with open(real_hub_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=sla_header)
    writer.writeheader()
    writer.writerows(kpi_rdd.collect())

with open(real_rejected_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rejected_header)
    writer.writeheader()
    writer.writerows(rejected_delivery_rdd.collect())

print(f"Success! Your file is saved at: {real_hub_path}")
print(f"Success! Your file is saved at: {real_rejected_path}")


Cleaning up existing directory: C:\Users\suele\Documents\assignments\assignments\week6_assignments\Day4\output\generated\hub_sla_report
Success! Your file is saved at: C:\Users\suele\Documents\assignments\assignments\week6_assignments\Day4\output\generated\hub_sla_report\hub_sla_report.csv
Success! Your file is saved at: C:\Users\suele\Documents\assignments\assignments\week6_assignments\Day4\output\generated\rejected_delivery_events\rejected_delivery_events.csv


In [ ]:
print("=="*20)
with open(real_hub_path) as file:
   
    iter = csv.reader(file)
    for i in iter:
        
        print(i)
print("=="*20)
with open(real_rejected_path) as f:
    iter = csv.reader(f)
    for i in iter:
        
        print(i)
print("=="*20)

#confirmed no records lost

['city_id', 'city', 'region', 'manager', 'sla_target', 'delivered_count', 'on_time_count', 'delay_hours', 'delivery_charge', 'on_time_pct', 'average_delay', 'total_charge', 'sla_gap', 'target_status']
['HUB-CHE', 'Chennai', 'South', 'Arun', '90.0', '2', '8', '64', '10626.600000000002', '400.0', '32.0', '10626.600000000002', '-310.0', 'True']
['HUB-CBE', 'Coimbatore', 'South', 'Meena', '92.0', '2', '4', '86', '14009.61', '200.0', '43.0', '14009.61', '-108.0', 'True']
['HUB-BLR', 'Bengaluru', 'South', 'Divya', '93.0', '2', '3', '15', '2807.8100000000004', '150.0', '7.5', '2807.8100000000004', '-57.0', 'True']
['HUB-PUN', 'Pune', 'West', 'Rahul', '89.0', '2', '1', '41', '7195.98', '50.0', '20.5', '7195.98', '39.0', 'False']
['HUB-HYD', 'Hyderabad', 'South-Central', 'Suresh', '91.0', '2', '0', '33', '3780.42', '0.0', '16.5', '3780.42', '91.0', 'False']
['is_valid', 'record', 'raw_line', 'error_reasons']
['False', '', 'EVT-1008,2026-07-08,SHP-5008,,SAME_DAY,DELIVERED,72,107,758,1.53,1007.7'